# 02 — Data Preprocessing & Sensor Synchronization

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Section 10:** Notebook-first traceable data transformation.
> **Section 5:** Keep each source traceable to its original session.

### Objectives:
1. Load synchronized smartphone (`S-*.csv`) and vehicle CAN (`V-*.csv`) sessions from IO-VNBD.
2. Filter high-frequency engine vibration and road noise using zero-phase Butterworth filters.
3. Separate dynamic linear acceleration from specific force using gravity compensation.
4. Detect stationary zero-velocity periods (ZUPT) via rolling variance gating.
5. Transform geodetic WGS84 GPS measurements into local Cartesian East-North-Up (ENU) coordinates.

## 1. Imports & Configuration

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.frame_transform import geodetic_to_enu
from src.preprocessing.imu_preprocessor import IMUPreprocessor
from src.preprocessing.data_loader import IOVNBDLoader

plots_dir = PROJECT_ROOT / 'plots' / 'preprocessing'
plots_dir.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', font_scale=1.1)
print('Preprocessing modules imported successfully.')

Preprocessing modules imported successfully.


## 2. Load Real IO-VNBD Session

In [2]:
loader = IOVNBDLoader()
available_sessions = loader.get_session_names()
print(f'Total sessions available: {len(available_sessions)}')
print(f'Sample sessions: {available_sessions[:8]}')

# Load primary training session 'M' (Driver B)
session_id = 'M'
sess = loader.load_session(session_id, preprocess_imu=True)

print(f'\n--- Session {session_id} Summary ---')
print(f'Driver          : {sess["driver"]}')
print(f'Split           : {sess["split"]}')
print(f'Duration        : {sess["time_s"][-1]:.1f} seconds ({sess["length"]:,} samples @ 10 Hz)')
print(f'Stationary (ZUPT): {sess["zupt_mask"].sum():,} samples ({sess["zupt_mask"].mean()*100:.1f}%)')
if sess["vehicle"]["speed_mps"] is not None:
    max_spd = np.nanmax(sess["vehicle"]["speed_mps"]) * 3.6
    print(f'Max Vehicle Speed: {max_spd:.1f} km/h')

Total sessions available: 72
Sample sessions: ['M', 'S1', 'S2', 'S3a', 'S3b', 'S3c', 'S4', 'Vfa01']

--- Session M Summary ---
Driver          : M (Driver B)
Split           : train
Duration        : 6171.7 seconds (105,974 samples @ 10 Hz)
Stationary (ZUPT): 34,774 samples (32.8%)
Max Vehicle Speed: 100.8 km/h


## 3. Vibration Filtering & Shock Suppression Analysis

In [3]:
# Visualize a 30-second window (300 samples) of raw vs filtered acceleration
w_start = 1000
w_end   = 1300
t_slice = sess['time_s'][w_start:w_end] - sess['time_s'][w_start]

fig, axs = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes_names = ['X (Pitch axis)', 'Y (Roll axis)', 'Z (Vertical axis)']

for i in range(3):
    axs[i].plot(t_slice, sess['accel_raw'][w_start:w_end, i], color='silver', label='Raw Sensor' if i==0 else '')
    axs[i].plot(t_slice, sess['accel_filtered'][w_start:w_end, i], color='#1f77b4', linewidth=1.5, label='Butterworth Filtered' if i==0 else '')
    axs[i].set_ylabel(f'{axes_names[i]}\n[m/s²]')
    axs[i].legend(loc='upper right')

axs[-1].set_xlabel('Time [seconds]')
fig.suptitle(f'IO-VNBD Session {session_id} — High-Frequency Vibration Suppression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(plots_dir / f'accel_filtering_{session_id}.png', dpi=200)
plt.close()
print(f'Vibration plot saved to: plots/preprocessing/accel_filtering_{session_id}.png')

Vibration plot saved to: plots/preprocessing/accel_filtering_M.png


<string>:13: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


## 4. Gravity Separation & Dynamic Acceleration

In [4]:
# Plot dynamic linear acceleration vs total specific force
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_slice, np.linalg.norm(sess['accel_raw'][w_start:w_end], axis=1), label='Total Specific Force (incl. gravity)', color='orange', alpha=0.7)
ax.plot(t_slice, np.linalg.norm(sess['accel_linear'][w_start:w_end], axis=1), label='Dynamic Linear Acceleration', color='#2ca02c', linewidth=1.8)
ax.axhline(9.80665, color='gray', linestyle='--', label='1g Earth Gravity (~9.81 m/s²)')
ax.set_ylabel('Acceleration Norm [m/s²]')
ax.set_xlabel('Time [seconds]')
ax.set_title('Gravity Vector Separation (Specific Force -> Dynamic Linear Acceleration)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(plots_dir / f'gravity_separation_{session_id}.png', dpi=200)
plt.close()
print(f'Gravity separation plot saved to: plots/preprocessing/gravity_separation_{session_id}.png')

Gravity separation plot saved to: plots/preprocessing/gravity_separation_M.png


## 5. Local Cartesian ENU Trajectory Reconstruction

In [5]:
enu = sess['enu_coords']
valid = (enu[:, 0] != 0) | (enu[:, 1] != 0)
enu_valid = enu[valid]

fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(enu_valid[:, 0], enu_valid[:, 1], c=sess['time_s'][valid], cmap='viridis', s=1.5, label='GNSS Path')
ax.plot(0, 0, 'r*', markersize=14, label='Start Reference Origin')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Elapsed Time [seconds]')
ax.set_xlabel('East Position [meters]')
ax.set_ylabel('North Position [meters]')
ax.axis('equal')
ax.set_title(f'Real Vehicle Trajectory (Local ENU Frame) — Session {session_id}')
ax.legend(loc='best')
plt.tight_layout()
plt.savefig(plots_dir / f'enu_trajectory_{session_id}.png', dpi=200)
plt.close()
print(f'ENU trajectory plot saved to: plots/preprocessing/enu_trajectory_{session_id}.png')

ENU trajectory plot saved to: plots/preprocessing/enu_trajectory_M.png


## 6. Preprocessing Verification & Sanity Metrics

In [6]:
# Compute key metrics
dt = np.diff(sess['time_s'])
median_dt = np.median(dt)
std_dt = np.std(dt)

print('=' * 60)
print('PREPROCESSING VERIFICATION REPORT')
print('=' * 60)
print(f'Sample Count          : {sess["length"]:,}')
print(f'Sampling Rate         : {1.0/median_dt:.2f} Hz (dt={median_dt*1000:.1f} ms, std={std_dt*1000:.2f} ms)')
print(f'Total Distance (GNSS) : {np.sum(np.linalg.norm(np.diff(enu_valid[:, :2], axis=0), axis=1)):.1f} meters')
print(f'Stationary Detection  : {sess["zupt_mask"].sum():,} samples ({sess["zupt_mask"].mean()*100:.1f}%)')
print(f'Linear Accel Mean Norm: {np.mean(np.linalg.norm(sess["accel_linear"], axis=1)):.3f} m/s² (expected near 0 when stationary)')
print('=' * 60)
print('Preprocessing notebook completed successfully.')

PREPROCESSING VERIFICATION REPORT
Sample Count          : 105,974
Sampling Rate         : 10.00 Hz (dt=100.0 ms, std=13598.53 ms)
Total Distance (GNSS) : 102010.4 meters
Stationary Detection  : 34,774 samples (32.8%)
Linear Accel Mean Norm: 1.353 m/s² (expected near 0 when stationary)
Preprocessing notebook completed successfully.
